The goal of this project is to compare different machine learning models for predicting dry bean varieties. I will compare accuracy, speed, overfitting, and maintainability to decide which model would be best for a production prototype.

In [1]:
import pandas as pd
import numpy as np
import time

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

from sklearn.metrics import accuracy_score, classification_report
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score

import matplotlib.pyplot as plt

print("Imports successful!")

Imports successful!


The dataset contains 13,611 beans with 16 measurements. The target is the Class column, which contains seven different bean varieties.

In [2]:
df = pd.read_excel("Dry_Bean_Dataset.xlsx")

print(df.shape)
df.head()

(13611, 17)


,Area,Perimeter,MajorAxisLength,MinorAxisLength,AspectRation,Eccentricity,ConvexArea,EquivDiameter,Extent,Solidity,roundness,Compactness,ShapeFactor1,ShapeFactor2,ShapeFactor3,ShapeFactor4,Class
0,28395,610.291,208.178117,173.888747,1.197191,0.549812,28715,190.141097,0.763923,0.988856,0.958027,0.913358,0.007332,0.003147,0.834222,0.998724,SEKER
1,28734,638.018,200.524796,182.734419,1.097356,0.411785,29172,191.272750,0.783968,0.984986,0.887034,0.953861,0.006979,0.003564,0.909851,0.998430,SEKER
2,29380,624.110,212.826130,175.931143,1.209713,0.562727,29690,193.410904,0.778113,0.989559,0.947849,0.908774,0.007244,0.003048,0.825871,0.999066,SEKER
3,30008,645.884,210.557999,182.516516,1.153638,0.498616,30724,195.467062,0.782681,0.976696,0.903936,0.928329,0.007017,0.003215,0.861794,0.994199,SEKER
4,30140,620.134,201.847882,190.279279,1.060798,0.333680,30417,195.896503,0.773098,0.990893,0.984877,0.970516,0.006697,0.003665,0.941900,0.999166,SEKER


In [3]:
print(df.isnull().sum().sum())
print(df["Class"].value_counts())

0
Class
DERMASON    3546
SIRA        2636
SEKER       2027
HOROZ       1928
CALI        1630
BARBUNYA    1322
BOMBAY       522
Name: count, dtype: int64


In [6]:
X = df.drop("Class", axis=1)
y = df["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_test.shape)

(10888, 16)
(2723, 16)


I will compare Logistic Regression, Decision Tree, Random Forest, and Gradient Boosting. I will record training accuracy, test accuracy, and training time for each model.

In [4]:
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=2000))
    ]),
    
    "Decision Tree": DecisionTreeClassifier(
        max_depth=10,
        random_state=42
    ),
    
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ),
    
    "Gradient Boosting": HistGradientBoostingClassifier(
        random_state=42
    )
}

In [7]:
results = []

for name, model in models.items():
    start = time.perf_counter()
    model.fit(X_train, y_train)
    train_time = time.perf_counter() - start

    train_acc = model.score(X_train, y_train)
    test_acc = model.score(X_test, y_test)

    results.append({
        "Model": name,
        "Training Accuracy": train_acc,
        "Test Accuracy": test_acc,
        "Training Time (sec)": train_time
    })

results_df = pd.DataFrame(results)
results_df

,Model,Training Accuracy,Test Accuracy,Training Time (sec)
0,Logistic Regression,0.926249,0.921410,0.689361
1,Decision Tree,0.963354,0.907088,0.531284
2,Random Forest,1.000000,0.919941,3.833192
3,Gradient Boosting,0.988611,0.924348,6.975785


The Random Forest shows the most overfitting because it reached 100% training accuracy but about 92% test accuracy. The Decision Tree also had a noticeable gap. Logistic Regression had very similar training and test scores, showing that it generalized well. Gradient Boosting had the highest test accuracy, but its training score was also much higher than its test score.

In [8]:
speed_results = []

for name, model in models.items():
    start = time.perf_counter()
    model.predict(X_test)
    predict_time = time.perf_counter() - start

    speed_results.append({
        "Model": name,
        "Prediction Time (sec)": predict_time
    })

speed_df = pd.DataFrame(speed_results)
speed_df

,Model,Prediction Time (sec)
0,Logistic Regression,0.012210
1,Decision Tree,0.004421
2,Random Forest,0.162148
3,Gradient Boosting,0.056099


I used K-Means to look for natural groups without giving the model the bean variety labels. I scaled the measurements first so features with larger values would not control the clustering.

In [10]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

kmeans = KMeans(
    n_clusters=7,
    random_state=42,
    n_init=10
)

clusters = kmeans.fit_predict(X_scaled)

print("Clusters found:", np.unique(clusters))
print("Silhouette Score:", silhouette_score(X_scaled, clusters))
print("Adjusted Rand Score:", adjusted_rand_score(y, clusters))

Clusters found: [0 1 2 3 4 5 6]
Silhouette Score: 0.3093797904374158
Adjusted Rand Score: 0.6686956744848731


K-Means found useful natural patterns without using the Class labels. The Adjusted Rand Score was about 0.67. Some clusters were strongly connected to certain varieties, such as BOMBAY, SEKER, HOROZ, and DERMASON. Other varieties had more overlap between clusters. This shows that the measurements contain useful natural structure, but the clusters should not automatically be treated as actual bean varieties.

In [12]:
cluster_table = pd.crosstab(
    clusters,
    y,
    rownames=["Cluster"],
    colnames=["Actual Class"]
)

cluster_table

Actual Class,BARBUNYA,BOMBAY,CALI,DERMASON,HOROZ,SEKER,SIRA
Cluster,,,,,,,
0,1142,1,1303,0,29,0,5
1,114,0,25,499,49,98,2326
2,18,0,2,115,0,1874,24
3,8,0,30,7,1661,0,63
4,0,0,0,2912,2,52,191
5,0,520,0,0,0,0,0
6,40,1,270,13,187,3,27


In [13]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_results = []

for name, model in models.items():
    scores = cross_val_score(model, X, y, cv=cv, scoring="accuracy")

    cv_results.append({
        "Model": name,
        "CV Mean Accuracy": scores.mean(),
        "CV Std": scores.std()
    })

cv_df = pd.DataFrame(cv_results)
cv_df

,Model,CV Mean Accuracy,CV Std
0,Logistic Regression,0.923812,0.005970
1,Decision Tree,0.906693,0.004534
2,Random Forest,0.925796,0.004769
3,Gradient Boosting,0.927853,0.003925


I used 5-fold stratified cross-validation to compare the models more fairly. This keeps the bean varieties represented in each fold and helps show how consistently each model performs.

All four models performed reasonably well. Logistic Regression had strong accuracy while being simple and fast. The Decision Tree performed the worst and showed some overfitting. Random Forest performed well but reached 100% training accuracy, showing overfitting. Gradient Boosting had the highest test and cross-validation accuracy, but it required more training time.
Overall, Logistic Regression gives a good balance between accuracy, speed, simplicity, and maintainability.

In [14]:
final_results = results_df.merge(speed_df, on="Model")
final_results = final_results.merge(cv_df, on="Model")

final_results.round(4)

,Model,Training Accuracy,Test Accuracy,Training Time (sec),Prediction Time (sec),CV Mean Accuracy,CV Std
0,Logistic Regression,0.9262,0.9214,0.6894,0.0122,0.9238,0.0060
1,Decision Tree,0.9634,0.9071,0.5313,0.0044,0.9067,0.0045
2,Random Forest,1.0000,0.9199,3.8332,0.1621,0.9258,0.0048
3,Gradient Boosting,0.9886,0.9243,6.9758,0.0561,0.9279,0.0039


I would consider Logistic Regression for the first production prototype. It reached about 92% accuracy and performed consistently during cross-validation. It was also faster to train and showed less overfitting than the more complex models. Gradient Boosting had slightly better accuracy, so it would also be worth testing. However, Logistic Regression is simpler to explain, maintain, and retrain.This classroom model is not ready to control real sorting equipment. More testing with real production data would be needed first.

The bean measurements were useful for both classification and clustering. Supervised models reached around 90–93% accuracy, while K-Means also found some natural groups without using the labels. Overall, the results show that machine learning could be useful for a dry bean sorting prototype.